<a href="https://colab.research.google.com/github/cuppsmil/Project/blob/main/%22reports-with-google-sheets%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 2 кейс

**Соберите данные из Google Sheets и напишите функцию `generate_report`, которая принимает три списка (данные с каждого листа в Google Sheets) и сохраняет отчет в файл `student_debt_report.txt`**

**Важно**

Перед началом решения задачи выполните следующую ячейку - в ней скачиваются нужные файлы

In [1]:
!wget https://gist.github.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json

!wget https://gist.github.com/Vs8th/39c5deed0f5539d781f00328f7fd4fe0/raw/result.txt

--2026-05-12 16:20:16--  https://gist.github.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json
Resolving gist.github.com (gist.github.com)... 140.82.113.3
Connecting to gist.github.com (gist.github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json [following]
--2026-05-12 16:20:16--  https://gist.githubusercontent.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2358 (2.3K) [text/plain]
Saving to: ‘creds.json’

creds.json          100%[===================>]   2.30K  --.-KB/s    in 0s      

2026-05-12 16:20:16 (41.7 MB/s) - ‘creds.json’ saved [2

Чтобы посмотреть как выглядят сами гугл таблицы в оригинале - можете перейти по [ссылке](https://docs.google.com/spreadsheets/d/1hRnw-PEftF0J-6KY7InlILVwWdkJu4vJiGwRjuE_P4U/edit?usp=sharing) и изучить.

Небольшое описание столбцов в таблицах:  
**Лист1:** `student_id` - айди студента; `student_name` - имя студента; `installment` (Y/N) - факт наличия рассрочки у студента (Y - рассрочка есть, N - рассрочки нет).  
**Лист2:** `student_id` - айди студента; `last_payment_date` - дата последней полученной оплаты; `expected_payment_date` - дата, в которую ожидаем следующий платеж (рассчитывается от даты последнего платежа).  
**Лист3:** `student_id` - айди студента; `already_payed_amount` - уже выплаченная часть рассрочки; `left_to_pay` - оставшаяся (невыплаченная) часть; `one-time_payment` - единоразовый платеж; `installment_amount` - общая сумма, которая бралась в рассрочку.

Как примерно должен выглядеть результат:

In [ ]:
with open('result.txt') as f:
  print(f.read())

Студент Иванов У.У. - долг 100000 рублей
Студент Петров Е.Е. - долг 250000 рублей
Студент Сидоров И.И. - долг 10000 рублей


In [2]:
#@title Если возникнут трудности с определением `scope`, подсказка:
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import pprint
scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
         'https://www.googleapis.com/auth/drive']
creds = ServiceAccountCredentials.from_json_keyfile_name('creds.json', scope)

# Авторизуемся в Google Sheets API
client = gspread.authorize(creds)

sheet1 = client.open("Installments").worksheet("Лист1")
sheet1_data = sheet1.get_all_records()
sheet2 = client.open("Installments").worksheet("Лист2")
sheet2_data = sheet2.get_all_records()
sheet3 = client.open("Installments").worksheet("Лист3")
sheet3_data = sheet3.get_all_records()

**Примечание**

Считать должников необходимо на 1 марта 2023 года. То есть определяя количество просроченных платежей, мы определяем их количество не на настоящую дату, а на **1 марта 2023 года**. А периоды внесения платежей для всех студентов одинаковы - **6 месяцев** (183 дня).

То есть, если ожидаемый платеж должен был пройти 3 февраля 2022 года, то к 1 марта 2023 студент просрочил уже три платежа (3 февраля 2022, 3 августа 2022 и 3 февраля 2023). При расчетах отталкивайтесь от даты ожидаемого платежа, разницу между платежами берите 183 дня.

**Решение**

Напишите свое решение ниже

In [60]:
from datetime import datetime, timedelta

def generate_report(sheet1_data, sheet2_data, sheet3_data):
  REFERENCE_DATE = datetime(2023, 3, 1)
  PAYMENT_PERIOD_DAYS = 183

    #Собираем студентов с рассрочкой
  inst_students = {}
  for row in sheet1_data:
      sid = str(row.get('student_id', '')).strip()
      installment = str(row.get('installment', '')).strip().upper()
      if sid and installment == 'Y':
          inst_students[sid] = str(row.get('student_name', '')).strip()

    #Собираем ожидаемые даты платежей
  expected_dates = {}
  for row in sheet2_data:
      sid = str(row.get('student_id', '')).strip()
      if sid and row.get('expected_payment_date'):
          expected_dates[sid] = row['expected_payment_date']

    #Собираем финансовые данных
  payment_info = {}
  for row in sheet3_data:
      sid = str(row.get('student_id', '')).strip()
      if sid:
          try:
              otp = float(str(row.get('one-time_payment', 0)).replace(',', '.'))
          except (ValueError, TypeError):
              otp = 0.0
          try:
              ltp = float(str(row.get('left_to_pay', 0)).replace(',', '.'))
          except (ValueError, TypeError):
              ltp = 0.0
          payment_info[sid] = {'one_time': otp, 'left': ltp}

    #Рассчитываем задолженности
  debts = []
  for student_id, student_name in inst_students.items():
      if student_id not in expected_dates or student_id not in payment_info:
          continue

      #Парсинг даты ожидаемого платежа
      exp_raw = expected_dates[student_id]
      if isinstance(exp_raw, datetime):
          exp_date = exp_raw
      elif isinstance(exp_raw, str):
          date_str = exp_raw.strip().split()[0]
          try:
              exp_date = datetime.strptime(date_str, '%d.%m.%Y')
          except ValueError:
              try:
                  exp_date = datetime.strptime(date_str, '%Y-%m-%d')
              except ValueError:
                  continue
      else:
          continue

      #Если первый платёж ещё не наступил, долга нет
      if exp_date > REFERENCE_DATE:
          continue

      #Считаем количество просроченных платежей
      missed_count = 0
      current_date = exp_date
      while current_date <= REFERENCE_DATE:
          missed_count += 1
          current_date += timedelta(days=PAYMENT_PERIOD_DAYS)

      if missed_count > 0:
          info = payment_info[student_id]
          raw_debt = missed_count * info['one_time']

         #долг не может быть больше оставшейся суммы
          final_debt = min(raw_debt, info['left'])
          if final_debt > 0:
              debts.append((student_name, int(round(final_debt))))

  #Записываем результат в файл
  with open('student_debt_report.txt', 'w', encoding='utf-8') as f:
      for name, debt in debts:
          f.write(f"Студент {name} - долг {debt} рублей\n")



In [61]:
#Загрузка данных
sheet1_data = client.open("Installments").worksheet("Лист1").get_all_records()
sheet2_data = client.open("Installments").worksheet("Лист2").get_all_records()
sheet3_data = client.open("Installments").worksheet("Лист3").get_all_records()

#Генерация отчёта
generate_report(sheet1_data, sheet2_data, sheet3_data)

#Сравнение с эталоном
with open('student_debt_report.txt', 'r', encoding='utf-8') as f:
    user = set(line.strip() for line in f if line.strip())
with open('student_debt.txt', 'r', encoding='utf-8') as f:
    correct = set(line.strip() for line in f if line.strip())
if user == correct:
    print("\nОтчёт верный! Все задолженности найдены.")
else:
    print("\nРасхождения:")
    print("Лишние строки:", user - correct)
    print("Недостающие строки:", correct - user)


📋 Содержимое student_debt_report.txt:
Студент Кузнецова К.А. - долг 266666 рублей
Студент Петров К.А. - долг 66666 рублей
Студент Иванов А.А. - долг 100000 рублей
Студент Иванов П.П. - долг 116666 рублей
Студент Петров И.И. - долг 116666 рублей
...

🎉 Отчёт верный! Все задолженности найдены.


✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [64]:
# Здесь будет скачиваться файл с эталонным ответом

!wget https://gist.github.com/Vs8th/63832f093f4db8d8b251ba5d39571f3d/raw/student_debt.txt

import pandas as pd

user_answer = pd.read_csv('student_debt_report.txt')
correct_answer = pd.read_csv('student_debt.txt')



--2026-05-12 17:33:28--  https://gist.github.com/Vs8th/63832f093f4db8d8b251ba5d39571f3d/raw/student_debt.txt
Resolving gist.github.com (gist.github.com)... 140.82.112.3
Connecting to gist.github.com (gist.github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/63832f093f4db8d8b251ba5d39571f3d/raw/student_debt.txt [following]
--2026-05-12 17:33:28--  https://gist.githubusercontent.com/Vs8th/63832f093f4db8d8b251ba5d39571f3d/raw/student_debt.txt
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 11325 (11K) [text/plain]
Saving to: ‘student_debt.txt.3’

student_debt.txt.3  100%[===================>]  11.06K  --.-KB/s    in 0s      

2026-05-12 17:33:28 (64.8 MB/

In [65]:
try:
  assert (user_answer == correct_answer).all().all(), 'Ответы не совпадают'
  assert user_answer.columns.equals(correct_answer.columns), 'Названия столбцов не совпадают'
except Exception as err:
  raise AssertionError(f'При проверке возникла ошибка {repr(err)}')
else:
  print('Поздравляем, Вы справились и успешно прошли все проверки!!')

Поздравляем, Вы справились и успешно прошли все проверки!!
